# **Preparation of Trees**

## **Packages**

In [15]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import laspy
import geopandas as gpd

from scipy.spatial import cKDTree
import open3d as o3d

## **Paths**

In [22]:
# Hier später nur den Plotnamen ändern
PLOT_NAME = "BO1"


# Grundordner
RCT_BASE_DIR = Path(r"Z:\Ghana\RCT_outputs")
STEMMAP_BASE_DIR = Path(r"Z:\Ghana\Stemmaps")


# Eingabedatei mit allen RCT-vorsegmentierten Bäumen
RCT_LAZ_PATH = (
    RCT_BASE_DIR
    / f"{PLOT_NAME}_preseg_RCT"
    / "output"
    / "rct"
    / "segmented"
    / "merged.laz"
)


STEMMAP_PATH = Path(
    r"C:\Users\laudenb\Documents\BO1_SM_copy\BO1_SM_copy.shp"
)


# Ein gemeinsamer Ausgabeordner
OUTPUT_DIR = (
    RCT_BASE_DIR
    / f"{PLOT_NAME}_preseg_RCT"
    / "tree_selection"
)


# Ausgabedateien innerhalb desselben Ordners
TREE_CATALOG_PATH = OUTPUT_DIR / f"{PLOT_NAME}_tree_catalog.csv"
SELECTION_TABLE_PATH = OUTPUT_DIR / f"{PLOT_NAME}_selected_trees.csv"
SELECTED_TREES_PATH = OUTPUT_DIR / f"{PLOT_NAME}_selected_trees.laz"


# Auswahlparameter
MIN_TREE_HEIGHT = 8.0
NUMBER_OF_HEIGHT_CLASSES = 15
TREES_PER_HEIGHT_CLASS = 8
RANDOM_SEED = 42


# Ausgabeordner erstellen
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## **Height estimation**

In [17]:
# =========================================================
# 3. BAUMHÖHEN AUS DER RCT-DATEI BERECHNEN
# =========================================================

CHUNK_SIZE = 5_000_000

chunk_summaries = []

with laspy.open(RCT_LAZ_PATH) as laz_file:

    dimensions = list(laz_file.header.point_format.dimension_names)

    if "PredInstance" not in dimensions:
        raise ValueError(
            "Die Dimension 'PredInstance' wurde in der LAZ-Datei nicht gefunden."
        )

    for chunk_number, points in enumerate(
        laz_file.chunk_iterator(CHUNK_SIZE),
        start=1
    ):
        instance_ids = np.asarray(points["PredInstance"])
        z_values = np.asarray(points.z)

        chunk_df = pd.DataFrame({
            "PredInstance": instance_ids,
            "z": z_values
        })

        chunk_summary = (
            chunk_df
            .groupby("PredInstance", as_index=False)
            .agg(
                z_min=("z", "min"),
                z_max=("z", "max"),
                number_of_points=("z", "size")
            )
        )

        chunk_summaries.append(chunk_summary)

        print(f"Block {chunk_number} verarbeitet")


# Ergebnisse aller Blöcke zusammenführen
tree_catalog = (
    pd.concat(chunk_summaries, ignore_index=True)
    .groupby("PredInstance", as_index=False)
    .agg(
        z_min=("z_min", "min"),
        z_max=("z_max", "max"),
        number_of_points=("number_of_points", "sum")
    )
)


# Segmenthöhe berechnen
tree_catalog["tree_height"] = (
    tree_catalog["z_max"] - tree_catalog["z_min"]
)


# Nach Höhe sortieren
tree_catalog = (
    tree_catalog
    .sort_values("tree_height")
    .reset_index(drop=True)
)


tree_catalog

Block 1 verarbeitet
Block 2 verarbeitet
Block 3 verarbeitet
Block 4 verarbeitet
Block 5 verarbeitet
Block 6 verarbeitet
Block 7 verarbeitet
Block 8 verarbeitet
Block 9 verarbeitet
Block 10 verarbeitet
Block 11 verarbeitet
Block 12 verarbeitet
Block 13 verarbeitet
Block 14 verarbeitet
Block 15 verarbeitet
Block 16 verarbeitet
Block 17 verarbeitet
Block 18 verarbeitet
Block 19 verarbeitet
Block 20 verarbeitet
Block 21 verarbeitet
Block 22 verarbeitet
Block 23 verarbeitet
Block 24 verarbeitet
Block 25 verarbeitet
Block 26 verarbeitet
Block 27 verarbeitet
Block 28 verarbeitet
Block 29 verarbeitet
Block 30 verarbeitet
Block 31 verarbeitet
Block 32 verarbeitet
Block 33 verarbeitet
Block 34 verarbeitet
Block 35 verarbeitet
Block 36 verarbeitet
Block 37 verarbeitet
Block 38 verarbeitet
Block 39 verarbeitet
Block 40 verarbeitet
Block 41 verarbeitet
Block 42 verarbeitet
Block 43 verarbeitet
Block 44 verarbeitet
Block 45 verarbeitet
Block 46 verarbeitet
Block 47 verarbeitet
Block 48 verarbeitet
B

,PredInstance,z_min,z_max,number_of_points,tree_height
0,6500,258.11,258.62,1561,0.51
1,9847,258.89,259.57,2429,0.68
2,2307,256.89,257.73,1170,0.84
3,2107,260.52,261.39,2352,0.87
4,5872,258.16,259.03,5697,0.87
...,...,...,...,...,...
12426,5222,259.40,303.49,2033312,44.09
12427,7139,260.16,304.62,1199720,44.46
12428,9190,259.09,305.99,1481388,46.90
12429,5267,254.86,302.93,1035315,48.07


In diesem Schritt wird die große RCT-Punktwolke blockweise eingelesen, damit nicht die gesamte LAZ-Datei gleichzeitig im Arbeitsspeicher liegen muss. Für jede PredInstance werden die minimale und maximale Z-Koordinate sowie die Punktanzahl bestimmt. Anschließend werden die Ergebnisse aller Blöcke zusammengeführt und daraus die vertikale Segmenthöhe jedes vorsegmentierten Baumes berechnet.

## **Excluding small Elements**

In [18]:
number_before = len(tree_catalog)

# Ungültige RCT-IDs entfernen
tree_catalog_filtered = tree_catalog[
    tree_catalog["PredInstance"] > 0
].copy()

number_after_id_filter = len(tree_catalog_filtered)

# Mindesthöhe anwenden
tree_catalog_filtered = tree_catalog_filtered[
    tree_catalog_filtered["tree_height"] >= MIN_TREE_HEIGHT
].copy()

tree_catalog_filtered = (
    tree_catalog_filtered
    .sort_values("tree_height")
    .reset_index(drop=True)
)

number_after_height_filter = len(tree_catalog_filtered)


print(f"RCT-Segmente insgesamt: {number_before}")
print(
    f"Nach Entfernung ungültiger PredInstance-IDs: "
    f"{number_after_id_filter}"
)
print(
    f"Nach Mindesthöhe von {MIN_TREE_HEIGHT:.1f} m: "
    f"{number_after_height_filter}"
)

print("\nHöhenbereich der verbleibenden Segmente:")
print(
    f"{tree_catalog_filtered['tree_height'].min():.2f} bis "
    f"{tree_catalog_filtered['tree_height'].max():.2f} m"
)

tree_catalog_filtered.head(10)

RCT-Segmente insgesamt: 12431
Nach Entfernung ungültiger PredInstance-IDs: 12430
Nach Mindesthöhe von 8.0 m: 1801

Höhenbereich der verbleibenden Segmente:
8.00 bis 48.07 m


,PredInstance,z_min,z_max,number_of_points,tree_height
0,3191,261.72,269.72,120543,8.00
1,6983,258.77,266.77,47726,8.00
2,11550,254.28,262.29,100096,8.01
3,9659,255.95,263.96,18434,8.01
4,8129,256.19,264.20,58509,8.01
5,2782,260.80,268.82,243366,8.02
6,8054,258.04,266.06,31219,8.02
7,11739,256.93,264.95,91538,8.02
8,84,252.79,260.82,48962,8.03
9,3876,259.68,267.71,177727,8.03


Hier werden zu kleine Bäum eoder Sträucher rausgefiltert. Als Grenze wird dabei ein Wert von 8m verwendet. Dieser Wert wurde aus den Inventurdaten ermittelt. Dort haben alle relevanten Bäume (DBH>10cm) eine Höhe von mindestens 9m. 

## **Connecting Stempoints and Trees**

In [24]:
# =========================================================
# 5. STAMMPOSITION BESTIMMEN UND STEMMAP ZUORDNEN
# =========================================================

SLICE_MIN, SLICE_MAX = 1.35, 1.65
MIN_SLICE_POINTS = 20
EXTRA_BUFFER = 0.40

valid_ids = tree_catalog_filtered["PredInstance"].astype(int).to_numpy()
z_min = tree_catalog_filtered.set_index("PredInstance")["z_min"].to_dict()
stem_points = {tree_id: [] for tree_id in valid_ids}

# Punkte im Bereich von 1,35–1,65 m sammeln
with laspy.open(RCT_LAZ_PATH) as laz_file:
    for chunk_number, points in enumerate(
        laz_file.chunk_iterator(CHUNK_SIZE), start=1
    ):
        ids = np.asarray(points["PredInstance"], dtype=int)
        mask = np.isin(ids, valid_ids)

        ids = ids[mask]
        xyz = np.column_stack((
            np.asarray(points.x)[mask],
            np.asarray(points.y)[mask],
            np.asarray(points.z)[mask]
        ))

        for tree_id in np.unique(ids):
            tree_xyz = xyz[ids == tree_id]
            relative_z = tree_xyz[:, 2] - z_min[tree_id]
            slice_xyz = tree_xyz[
                (relative_z >= SLICE_MIN) &
                (relative_z <= SLICE_MAX)
            ]

            if len(slice_xyz):
                stem_points[tree_id].append(slice_xyz[:, :2])

        print(f"Block {chunk_number} verarbeitet")


# Stemmap laden und ungültige Geometrien entfernen
stemmap = gpd.read_file(STEMMAP_PATH)

stemmap = stemmap[
    stemmap.geometry.notna()
    & ~stemmap.geometry.is_empty
].copy()

# MultiPoint-Geometrien in einzelne Points zerlegen
stemmap = stemmap.explode(index_parts=False).reset_index(drop=True)

# Kontrolle
print(stemmap.geometry.geom_type.value_counts())

# Nur echte Points behalten
stemmap = stemmap[stemmap.geometry.geom_type == "Point"].copy()

stemmap["x"] = stemmap.geometry.x
stemmap["y"] = stemmap.geometry.y

stemmap = stemmap[
    np.isfinite(stemmap["x"])
    & np.isfinite(stemmap["y"])
].reset_index(drop=True)

stem_xy = stemmap[["x", "y"]].to_numpy()
stem_tree = cKDTree(stem_xy)

print(f"Gültige Stemmap-Punkte: {len(stemmap)}")

results = []

for tree_id in valid_ids:
    arrays = stem_points[tree_id]

    if not arrays:
        results.append([tree_id, np.nan, np.nan, np.nan,
                        np.nan, "keine Stammpunkte"])
        continue

    xy = np.vstack(arrays)

    if len(xy) < MIN_SLICE_POINTS:
        results.append([tree_id, np.nan, np.nan, np.nan,
                        np.nan, "zu wenige Stammpunkte"])
        continue

    stem_x, stem_y = np.median(xy, axis=0)
    distances = np.linalg.norm(xy - [stem_x, stem_y], axis=1)
    search_radius = np.percentile(distances, 90) + EXTRA_BUFFER

    candidates = stem_tree.query_ball_point(
        [stem_x, stem_y],
        search_radius
    )

    if len(candidates) == 1:
        candidate = candidates[0]
        stemmap_id = stemmap.iloc[candidate]["id"]
        match_distance = np.linalg.norm(
            stem_xy[candidate] - [stem_x, stem_y]
        )
        status = "eindeutig"
    elif len(candidates) == 0:
        stemmap_id = np.nan
        match_distance = np.nan
        status = "kein Stemmap-Punkt"
    else:
        stemmap_id = np.nan
        match_distance = np.nan
        status = "mehrere Stemmap-Punkte"

    results.append([
        tree_id, stem_x, stem_y, search_radius,
        stemmap_id, status, match_distance
    ])


matches = pd.DataFrame(
    results,
    columns=[
        "PredInstance", "stem_x", "stem_y", "search_radius",
        "stemmap_id", "match_status", "match_distance"
    ]
)

tree_catalog_filtered = tree_catalog_filtered.merge(
    matches,
    on="PredInstance",
    how="left"
)

print(tree_catalog_filtered["match_status"].value_counts(dropna=False))

Block 1 verarbeitet
Block 2 verarbeitet
Block 3 verarbeitet
Block 4 verarbeitet
Block 5 verarbeitet
Block 6 verarbeitet
Block 7 verarbeitet
Block 8 verarbeitet
Block 9 verarbeitet
Block 10 verarbeitet
Block 11 verarbeitet
Block 12 verarbeitet
Block 13 verarbeitet
Block 14 verarbeitet
Block 15 verarbeitet
Block 16 verarbeitet
Block 17 verarbeitet
Block 18 verarbeitet
Block 19 verarbeitet
Block 20 verarbeitet
Block 21 verarbeitet
Block 22 verarbeitet
Block 23 verarbeitet
Block 24 verarbeitet
Block 25 verarbeitet
Block 26 verarbeitet
Block 27 verarbeitet
Block 28 verarbeitet
Block 29 verarbeitet
Block 30 verarbeitet
Block 31 verarbeitet
Block 32 verarbeitet
Block 33 verarbeitet
Block 34 verarbeitet
Block 35 verarbeitet
Block 36 verarbeitet
Block 37 verarbeitet
Block 38 verarbeitet
Block 39 verarbeitet
Block 40 verarbeitet
Block 41 verarbeitet
Block 42 verarbeitet
Block 43 verarbeitet
Block 44 verarbeitet
Block 45 verarbeitet
Block 46 verarbeitet
Block 47 verarbeitet
Block 48 verarbeitet
B

In diesem Schritt wird für jedes ausreichend hohe RCT-Segment die Stammposition auf ungefähr 1,5 m Höhe bestimmt. Dafür werden alle Punkte zwischen 1,35 und 1,65 m über dem niedrigsten Punkt des Segments verwendet und deren mittlere Lage über den Median der X- und Y-Koordinaten berechnet. Anschließend wird ein individueller Suchradius aus der räumlichen Verteilung dieser Punkte plus einem zusätzlichen Puffer von 0,40 m abgeleitet. Liegt innerhalb dieses Radius genau ein Stemmap-Punkt, wird dessen ID dem Segment eindeutig zugeordnet; bei keinem oder mehreren Punkten bleibt die Zuordnung offen.

## **Visual Control**

### **Assigned Trees**

In [25]:
# =========================================================
# ZUGEWIESENE BÄUME AUSGEDÜNNT ANZEIGEN
# =========================================================

import open3d as o3d

DISPLAY_EVERY_NTH_POINT = 10

# IDs der eindeutig zugewiesenen RCT-Bäume
assigned_ids = (
    tree_catalog_filtered.loc[
        tree_catalog_filtered["match_status"] == "eindeutig",
        "PredInstance"
    ]
    .astype(int)
    .to_numpy()
)

xyz_parts = []

# Punkte der zugewiesenen Bäume aus der LAZ-Datei laden
with laspy.open(RCT_LAZ_PATH) as laz_file:
    for points in laz_file.chunk_iterator(CHUNK_SIZE):

        ids = np.asarray(points["PredInstance"], dtype=int)
        mask = np.isin(ids, assigned_ids)

        if np.any(mask):
            xyz = np.column_stack((
                np.asarray(points.x)[mask],
                np.asarray(points.y)[mask],
                np.asarray(points.z)[mask]
            ))

            xyz_parts.append(xyz[::DISPLAY_EVERY_NTH_POINT])

if not xyz_parts:
    raise ValueError("Es wurden keine eindeutig zugewiesenen Baumpunkte gefunden.")

assigned_xyz_reduced = np.vstack(xyz_parts)

# Lokalen Ursprung setzen, damit Open3D mit den großen Koordinaten klarkommt
assigned_xyz_reduced -= assigned_xyz_reduced.min(axis=0)

cloud = o3d.geometry.PointCloud()
cloud.points = o3d.utility.Vector3dVector(assigned_xyz_reduced)

print(f"Zugewiesene Bäume: {len(assigned_ids)}")
print(f"Dargestellte Punkte: {len(assigned_xyz_reduced):,}")

o3d.visualization.draw_geometries(
    [cloud],
    window_name="Zugewiesene Bäume – ausgedünnt",
    width=1400,
    height=900
)

Zugewiesene Bäume: 552
Dargestellte Punkte: 24,402,870


### **Unassigned Trees**

In [26]:
# =========================================================
# NUR NICHT ZUGEWIESENE BÄUME AUSGEDÜNNT IN OPEN3D
# =========================================================

DISPLAY_EVERY_NTH_POINT = 20

unassigned_ids = (
    tree_catalog_filtered.loc[
        tree_catalog_filtered["match_status"] != "eindeutig",
        "PredInstance"
    ]
    .astype(int)
    .to_numpy()
)

xyz_parts = []

with laspy.open(RCT_LAZ_PATH) as laz_file:
    for points in laz_file.chunk_iterator(CHUNK_SIZE):

        ids = np.asarray(points["PredInstance"], dtype=int)
        mask = np.isin(ids, unassigned_ids)

        if np.any(mask):
            xyz = np.column_stack((
                np.asarray(points.x)[mask],
                np.asarray(points.y)[mask],
                np.asarray(points.z)[mask]
            ))

            xyz_parts.append(xyz[::DISPLAY_EVERY_NTH_POINT])

unassigned_xyz = np.vstack(xyz_parts)

# Lokalen Ursprung setzen
unassigned_xyz -= unassigned_xyz.min(axis=0)

cloud = o3d.geometry.PointCloud()
cloud.points = o3d.utility.Vector3dVector(unassigned_xyz)

print(f"Nicht zugewiesene Bäume: {len(unassigned_ids)}")
print(f"Dargestellte Punkte: {len(unassigned_xyz):,}")

o3d.visualization.draw_geometries(
    [cloud],
    window_name="Nicht zugewiesene Bäume",
    width=1400,
    height=900
)

Nicht zugewiesene Bäume: 1249
Dargestellte Punkte: 13,340,865


## **Manuel Assignment**

In [27]:
# Fehlende Variable inner_unassigned erzeugen

inner_unassigned = tree_catalog_filtered[
    tree_catalog_filtered["match_status"] != "eindeutig"
].copy()

inner_unassigned = inner_unassigned.dropna(
    subset=["stem_x", "stem_y"]
).copy()

distances, indices = stem_tree.query(
    inner_unassigned[["stem_x", "stem_y"]].to_numpy(),
    k=1
)

inner_unassigned["nearest_distance"] = distances
inner_unassigned["nearest_stemmap_index"] = indices

print(f"Nicht zugewiesene Bäume: {len(inner_unassigned)}")
print(inner_unassigned[
    ["PredInstance", "match_status", "nearest_distance"]
].head())

# =========================================================
# MANUELLE KONTROLLE DER NICHT ZUGEWIESENEN INNEREN BÄUME
# NUR BIS 1,5 M ENTFERNUNG ZUM NÄCHSTEN STEMMAP-PUNKT
# =========================================================

DISPLAY_EVERY_NTH_POINT = 20
NUMBER_OF_NEIGHBOR_TREES = 4
NUMBER_OF_STEMMAP_POINTS = 4
MAX_REVIEW_DISTANCE = 1.5

MANUAL_REVIEW_PATH = (
    OUTPUT_DIR / f"{PLOT_NAME}_manual_stemmap_review.csv"
)

# ---------------------------------------------------------
# 1. Zu prüfende Bäume vorbereiten
# ---------------------------------------------------------

review_trees = inner_unassigned[
    inner_unassigned["nearest_distance"] <= MAX_REVIEW_DISTANCE
].copy()

review_trees = (
    review_trees
    .sort_values("nearest_distance")
    .reset_index(drop=True)
)

review_ids = review_trees["PredInstance"].astype(int).to_numpy()

tree_positions = tree_catalog_filtered.dropna(
    subset=["stem_x", "stem_y"]
).copy()

tree_positions["PredInstance"] = (
    tree_positions["PredInstance"].astype(int)
)

all_tree_ids = tree_positions["PredInstance"].to_numpy()


# ---------------------------------------------------------
# 2. Vollständige Baumpunkte einmal laden und ausdünnen
# ---------------------------------------------------------

tree_points_by_id = {
    tree_id: []
    for tree_id in all_tree_ids
}

with laspy.open(RCT_LAZ_PATH) as laz_file:

    for chunk_number, points in enumerate(
        laz_file.chunk_iterator(CHUNK_SIZE),
        start=1
    ):
        ids = np.asarray(points["PredInstance"], dtype=int)

        relevant_mask = np.isin(ids, all_tree_ids)

        ids = ids[relevant_mask]

        xyz = np.column_stack((
            np.asarray(points.x)[relevant_mask],
            np.asarray(points.y)[relevant_mask],
            np.asarray(points.z)[relevant_mask]
        ))

        for tree_id in np.unique(ids):

            tree_xyz = xyz[ids == tree_id]

            tree_points_by_id[tree_id].append(
                tree_xyz[::DISPLAY_EVERY_NTH_POINT]
            )

        print(f"Block {chunk_number} verarbeitet")


for tree_id in list(tree_points_by_id):

    parts = tree_points_by_id[tree_id]

    if parts:
        tree_points_by_id[tree_id] = np.vstack(parts)
    else:
        tree_points_by_id[tree_id] = np.empty((0, 3))


# ---------------------------------------------------------
# 3. Bereits vorhandene Entscheidungen laden
# ---------------------------------------------------------

if MANUAL_REVIEW_PATH.exists():

    manual_results = pd.read_csv(MANUAL_REVIEW_PATH)

    already_reviewed = set(
        manual_results["PredInstance"].astype(int)
    )

    review_trees = review_trees[
        ~review_trees["PredInstance"]
        .astype(int)
        .isin(already_reviewed)
    ].reset_index(drop=True)

    decisions = manual_results.to_dict("records")

else:
    decisions = []


print(f"Noch zu kontrollierende Bäume: {len(review_trees)}")


# ---------------------------------------------------------
# 4. Hilfsfunktion für Kugeln
# ---------------------------------------------------------

def make_sphere(position, radius, color):

    sphere = o3d.geometry.TriangleMesh.create_sphere(
        radius=radius
    )

    sphere.translate(position)
    sphere.paint_uniform_color(color)
    sphere.compute_vertex_normals()

    return sphere


# ---------------------------------------------------------
# 5. Bäume einzeln anzeigen und mit J oder N bewerten
# ---------------------------------------------------------

for review_number, row in review_trees.iterrows():

    tree_id = int(row["PredInstance"])

    current_xyz = tree_points_by_id[tree_id]

    if len(current_xyz) == 0:
        continue

    current_xy = np.array([
        row["stem_x"],
        row["stem_y"]
    ])

    # Vier nächste andere RCT-Bäume
    other_trees = tree_positions[
        tree_positions["PredInstance"] != tree_id
    ].copy()

    other_trees["distance"] = np.sqrt(
        (other_trees["stem_x"] - current_xy[0]) ** 2
        +
        (other_trees["stem_y"] - current_xy[1]) ** 2
    )

    neighbor_ids = (
        other_trees
        .nsmallest(
            NUMBER_OF_NEIGHBOR_TREES,
            "distance"
        )["PredInstance"]
        .astype(int)
        .to_numpy()
    )

    # Vier nächste Stemmap-Punkte
    stem_distances, stem_indices = stem_tree.query(
        current_xy,
        k=min(NUMBER_OF_STEMMAP_POINTS, len(stemmap))
    )

    stem_indices = np.atleast_1d(stem_indices)
    stem_distances = np.atleast_1d(stem_distances)

    nearest_stem_index = int(stem_indices[0])
    nearest_stem_id = stemmap.iloc[nearest_stem_index]["id"]
    nearest_distance = float(stem_distances[0])

    origin = np.array([
        current_xy[0],
        current_xy[1],
        row["z_min"]
    ])

    geometries = []

    # Aktueller Baum: rot
    current_cloud = o3d.geometry.PointCloud()
    current_cloud.points = o3d.utility.Vector3dVector(
        current_xyz - origin
    )
    current_cloud.paint_uniform_color([0.9, 0.2, 0.1])
    geometries.append(current_cloud)

    # Vier Nachbarbäume: grau
    for neighbor_id in neighbor_ids:

        neighbor_xyz = tree_points_by_id[neighbor_id]

        if len(neighbor_xyz) == 0:
            continue

        neighbor_cloud = o3d.geometry.PointCloud()
        neighbor_cloud.points = o3d.utility.Vector3dVector(
            neighbor_xyz - origin
        )
        neighbor_cloud.paint_uniform_color([0.55, 0.55, 0.55])

        geometries.append(neighbor_cloud)

    # Berechnete Stammposition: blau
    calculated_stem_position = np.array([
        row["stem_x"],
        row["stem_y"],
        row["z_min"] + 1.5
    ]) - origin

    geometries.append(
        make_sphere(
            calculated_stem_position,
            radius=0.18,
            color=[0.0, 0.2, 1.0]
        )
    )

    # Stemmap-Punkte
    for position_number, stem_index in enumerate(stem_indices):

        stem_row = stemmap.iloc[int(stem_index)]

        stem_position = np.array([
            stem_row["x"],
            stem_row["y"],
            row["z_min"] + 1.5
        ]) - origin

        if position_number == 0:
            color = [0.0, 1.0, 0.0]
            radius = 0.25
        else:
            color = [1.0, 0.8, 0.0]
            radius = 0.18

        geometries.append(
            make_sphere(
                stem_position,
                radius=radius,
                color=color
            )
        )

    print(
        f"\nBaum {review_number + 1}/{len(review_trees)}"
        f" | PredInstance: {tree_id}"
        f" | nächster Stemmap-Punkt: {nearest_stem_id}"
        f" | Abstand: {nearest_distance:.2f} m"
    )

    print("J = zuordnen | N = ablehnen | Q = Kontrolle beenden")

    action = {"value": None}

    def accept_callback(vis):
        action["value"] = "zugewiesen"
        vis.close()
        return False

    def reject_callback(vis):
        action["value"] = "abgelehnt"
        vis.close()
        return False

    def quit_callback(vis):
        action["value"] = "beenden"
        vis.close()
        return False

    visualizer = o3d.visualization.VisualizerWithKeyCallback()

    visualizer.create_window(
        window_name=(
            f"PredInstance {tree_id} | "
            f"Stemmap {nearest_stem_id} | "
            f"{nearest_distance:.2f} m"
        ),
        width=1400,
        height=900
    )

    for geometry in geometries:
        visualizer.add_geometry(geometry)

    visualizer.register_key_callback(
        ord("J"),
        accept_callback
    )

    visualizer.register_key_callback(
        ord("N"),
        reject_callback
    )

    visualizer.register_key_callback(
        ord("Q"),
        quit_callback
    )

    visualizer.run()
    visualizer.destroy_window()

    if action["value"] == "beenden":
        print("Manuelle Kontrolle beendet.")
        break

    if action["value"] is None:
        action["value"] = "Fenster geschlossen"

    decisions.append({
        "PredInstance": tree_id,
        "manual_status": action["value"],
        "manual_stemmap_id": (
            nearest_stem_id
            if action["value"] == "zugewiesen"
            else np.nan
        ),
        "manual_distance": nearest_distance
    })

    pd.DataFrame(decisions).to_csv(
        MANUAL_REVIEW_PATH,
        index=False
    )


print("\nErgebnisse gespeichert unter:")
print(MANUAL_REVIEW_PATH)

Nicht zugewiesene Bäume: 1248
   PredInstance        match_status  nearest_distance
0          3191  kein Stemmap-Punkt          8.025348
1          6983  kein Stemmap-Punkt          2.756423
2         11550  kein Stemmap-Punkt          2.563140
3          9659  kein Stemmap-Punkt          1.456178
6          8054  kein Stemmap-Punkt          1.619677
Block 1 verarbeitet
Block 2 verarbeitet
Block 3 verarbeitet
Block 4 verarbeitet
Block 5 verarbeitet
Block 6 verarbeitet
Block 7 verarbeitet
Block 8 verarbeitet
Block 9 verarbeitet
Block 10 verarbeitet
Block 11 verarbeitet
Block 12 verarbeitet
Block 13 verarbeitet
Block 14 verarbeitet
Block 15 verarbeitet
Block 16 verarbeitet
Block 17 verarbeitet
Block 18 verarbeitet
Block 19 verarbeitet
Block 20 verarbeitet
Block 21 verarbeitet
Block 22 verarbeitet
Block 23 verarbeitet
Block 24 verarbeitet
Block 25 verarbeitet
Block 26 verarbeitet
Block 27 verarbeitet
Block 28 verarbeitet
Block 29 verarbeitet
Block 30 verarbeitet
Block 31 verarbeitet
Bloc

KeyboardInterrupt: 

In diesem Schritt werden nur bisher nicht eindeutig zugewiesene Bäume innerhalb der Stemmap-Fläche geprüft, deren nächster Stemmap-Punkt höchstens 1,5 m entfernt liegt. Für jeden Baum werden der aktuelle Baum, vier benachbarte Bäume sowie die vier nächstgelegenen Stemmap-Punkte in Open3D dargestellt. Mit J wird der nächstgelegene Stemmap-Punkt manuell bestätigt, mit N wird die Zuordnung abgelehnt und mit Q kann die Kontrolle beendet werden. Die Entscheidungen werden nach jedem Baum automatisch in einer CSV-Datei gespeichert, sodass die Prüfung später fortgesetzt werden kann.

## **Join of Inventory Data**

In [35]:
# =========================================================
# INVENTAR ZUORDNEN UND BÄUME MIT 15 HÖHENKLASSEN EXPORTIEREN
# =========================================================

INVENTORY_PATH = Path(
    r"Z:\Ghana\field data\Field Data Request.xlsx"
)

OUTPUT_DIR = Path(
    r"Z:\Ghana\RCT_outputs\BO1_preseg_RCT\output\rct\segmented"
)

TREE_EXPORT_DIR = OUTPUT_DIR / "BO1_Trees_Inventory"
TREE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

PLOT_TO_PID = {
    "BO1": "BOB-01",
}


# ---------------------------------------------------------
# 1. STEMMAP-ZUORDNUNGEN VORBEREITEN
# ---------------------------------------------------------

trees = tree_catalog_filtered.copy()
trees["plot"] = PLOT_NAME
trees["PredInstance"] = trees["PredInstance"].astype(int)

# Manuell bestätigte Zuordnungen übernehmen
if MANUAL_REVIEW_PATH.exists():
    manual = pd.read_csv(MANUAL_REVIEW_PATH)
    manual = manual[manual["manual_status"] == "zugewiesen"].copy()
    manual["PredInstance"] = manual["PredInstance"].astype(int)

    manual_map = manual.set_index(
        "PredInstance"
    )["manual_stemmap_id"]

    mask = trees["PredInstance"].isin(manual_map.index)

    trees.loc[mask, "stemmap_id"] = (
        trees.loc[mask, "PredInstance"].map(manual_map)
    )

    trees.loc[mask, "match_status"] = "manuell zugewiesen"


# ---------------------------------------------------------
# 2. IDS VEREINHEITLICHEN
# ---------------------------------------------------------

def normalize_id(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().replace(",", ".")

    try:
        number = float(text)
        return str(int(number)) if number.is_integer() else str(number)
    except ValueError:
        return text.upper()


# ---------------------------------------------------------
# 3. INVENTAR LADEN UND VERKNÜPFEN
# ---------------------------------------------------------

inventory = pd.read_excel(
    INVENTORY_PATH,
    sheet_name="Data"
)

inventory.columns = (
    inventory.columns.astype(str)
    .str.replace("\n", " ", regex=False)
    .str.strip()
)

trees["_pid"] = (
    trees["plot"]
    .map(PLOT_TO_PID)
    .astype("string")
    .str.strip()
    .str.upper()
)

trees["_tag"] = trees["stemmap_id"].apply(normalize_id)

inventory["_pid"] = (
    inventory["PID"]
    .astype("string")
    .str.strip()
    .str.upper()
)

inventory["_tag"] = inventory["Tag"].apply(normalize_id)

# Doppelte Plot-ID-Kombinationen nicht verwenden
inventory = inventory.dropna(
    subset=["_pid", "_tag"]
)

inventory = inventory[
    ~inventory.duplicated(
        ["_pid", "_tag"],
        keep=False
    )
].copy()

inventory = inventory.rename(
    columns={
        column: f"inventory_{column}"
        for column in inventory.columns
        if column not in [
            "_pid",
            "_tag",
            "PID",
            "Tag"
        ]
    }
)

trees = trees.merge(
    inventory,
    on=["_pid", "_tag"],
    how="left",
    validate="many_to_one",
)


# ---------------------------------------------------------
# 4. NUR ZUGEORDNETE BÄUME MIT INVENTARDATEN
# ---------------------------------------------------------

inventory_columns = [
    column
    for column in trees.columns
    if column.startswith("inventory_")
]

has_inventory = (
    trees[inventory_columns]
    .notna()
    .any(axis=1)
)

export_trees = trees[
    trees["match_status"].isin(
        ["eindeutig", "manuell zugewiesen"]
    )
    & has_inventory
].copy()

export_trees["PredInstance"] = (
    export_trees["PredInstance"].astype(int)
)

# ---------------------------------------------------------
# 5. 15 GLEICH BREITE HÖHENKLASSEN BERECHNEN
# ---------------------------------------------------------

export_trees["tree_height"] = pd.to_numeric(
    export_trees["tree_height"],
    errors="coerce"
)

if export_trees["tree_height"].isna().any():
    missing_heights = export_trees[
        export_trees["tree_height"].isna()
    ]["PredInstance"].tolist()

    raise ValueError(
        "Für folgende Bäume fehlt tree_height: "
        f"{missing_heights}"
    )
    
# tree_height sicher numerisch machen
export_trees["tree_height"] = pd.to_numeric(
    export_trees["tree_height"],
    errors="coerce"
)

# inf und -inf entfernen
export_trees["tree_height"] = export_trees[
    "tree_height"
].replace([np.inf, -np.inf], np.nan)

# Ungültige Höhen entfernen
export_trees = export_trees.dropna(
    subset=["tree_height"]
).copy()

if export_trees.empty:
    raise ValueError(
        "Keine Bäume mit gültiger tree_height für den Export vorhanden."
    )

min_height = float(export_trees["tree_height"].min())
max_height = float(export_trees["tree_height"].max())

if np.isclose(min_height, max_height):
    export_trees["Height_Class"] = 1
else:
    height_edges = np.linspace(
        min_height,
        max_height,
        16
    )

    export_trees["Height_Class"] = pd.cut(
        export_trees["tree_height"],
        bins=height_edges,
        labels=range(1, 16),
        include_lowest=True,
        right=True
    ).astype(int)

print(
    f"Höhenbereich: "
    f"{min_height:.2f} bis {max_height:.2f} m"
)

print("\nHöhenklassen:")
for class_number in range(1, 16):
    lower = height_edges[class_number - 1]
    upper = height_edges[class_number]

    count = (
        export_trees["Height_Class"]
        == class_number
    ).sum()

    print(
        f"Klasse {class_number:2d}: "
        f"{lower:.2f}–{upper:.2f} m | "
        f"{count} Bäume"
    )


# PredInstance → Height_Class
height_class_map = (
    export_trees
    .set_index("PredInstance")["Height_Class"]
    .astype(np.uint8)
    .to_dict()
)

export_ids = set(
    export_trees["PredInstance"]
)

print(f"\nBäume für den Export: {len(export_trees)}")


# ---------------------------------------------------------
# 6. SAUBERE MASTER-TABELLE ALS EXCEL SPEICHERN
# ---------------------------------------------------------

MASTER_EXCEL_PATH = (
    TREE_EXPORT_DIR / "KO4_tree_inventory_master.xlsx"
)

master_table = export_trees.drop(
    columns=["_pid", "_tag"],
    errors="ignore"
).copy()

# Technische inventory_-Präfixe entfernen
master_table.columns = [
    column.replace("inventory_", "")
    for column in master_table.columns
]

# Sinnvolle Spalten zuerst anzeigen
preferred_columns = [
    "PredInstance",
    "stemmap_id",
    "match_status",
    "tree_height",
    "Height_Class",
    "stem_x",
    "stem_y",
    "Tag",
    "PID",
    "Date",
    "Ecozone",
    "Site",
    "PT",
    "Subplot",
    "Tnb",
    "Species",
    "COR N",
    "COR W",
    "ELEV",
    "WD",
    "dbh (cm)",
    "d2",
    "H (m)",
    "H Mode",
    "Mass kg C",
    "Status",
    "Life Form",
]

preferred_columns = [
    column
    for column in preferred_columns
    if column in master_table.columns
]

remaining_columns = [
    column
    for column in master_table.columns
    if column not in preferred_columns
]

master_table = master_table[
    preferred_columns + remaining_columns
]

with pd.ExcelWriter(
    MASTER_EXCEL_PATH,
    engine="openpyxl"
) as writer:

    master_table.to_excel(
        writer,
        sheet_name="Tree Inventory",
        index=False
    )

    worksheet = writer.sheets["Tree Inventory"]

    # Kopfzeile fixieren
    worksheet.freeze_panes = "A2"

    # Filter aktivieren
    worksheet.auto_filter.ref = worksheet.dimensions

    # Sinnvolle Spaltenbreiten
    for column_cells in worksheet.columns:
        max_length = max(
            len(str(cell.value)) if cell.value is not None else 0
            for cell in column_cells
        )

        column_letter = column_cells[0].column_letter
        worksheet.column_dimensions[column_letter].width = min(
            max(max_length + 2, 10),
            35
        )

print("Saubere Excel-Tabelle gespeichert unter:")
print(MASTER_EXCEL_PATH)


# ---------------------------------------------------------
# 7. ALLE ZUGEORDNETEN BÄUME IN EINE LAZ SCHREIBEN
#    MIT ZUSÄTZLICHER DIMENSION Height_Class
# ---------------------------------------------------------

COMBINED_LAZ_PATH = (
    TREE_EXPORT_DIR
    / "KO4_Trees_Inventory.laz"
)

export_ids_array = np.asarray(
    sorted(export_ids),
    dtype=np.int64
)

written_points = 0

with laspy.open(RCT_LAZ_PATH) as source:

    header = source.header.copy()

    existing_dimensions = {
        dimension.name
        for dimension in header.point_format.dimensions
    }

    if "Height_Class" not in existing_dimensions:
        header.add_extra_dim(
            laspy.ExtraBytesParams(
                name="Height_Class",
                type=np.uint8,
                description="Tree height class from 1 to 15"
            )
        )

    with laspy.open(
        COMBINED_LAZ_PATH,
        mode="w",
        header=header
    ) as writer:

        for chunk_number, points in enumerate(
            source.chunk_iterator(CHUNK_SIZE),
            start=1
        ):
            ids = np.asarray(
                points["PredInstance"],
                dtype=np.int64
            )

            mask = np.isin(
                ids,
                export_ids_array
            )

            if np.any(mask):
                selected_points = points[mask]
                selected_ids = ids[mask]

                output_points = laspy.ScaleAwarePointRecord.zeros(
                    len(selected_points),
                    header=header
                )

                # Alle bestehenden Dimensionen übernehmen
                for dimension in source.header.point_format.dimension_names:
                    output_points[dimension] = (
                        selected_points[dimension]
                    )

                # Height_Class je Punkt über PredInstance zuweisen
                output_points["Height_Class"] = np.fromiter(
                    (
                        height_class_map[int(tree_id)]
                        for tree_id in selected_ids
                    ),
                    dtype=np.uint8,
                    count=len(selected_ids)
                )

                writer.write_points(output_points)

                written_points += len(output_points)

            print(
                f"\rBlock {chunk_number} verarbeitet | "
                f"{written_points:,} Punkte geschrieben",
                end=""
            )

print("\n\nExport abgeschlossen")
print(f"LAZ-Datei: {COMBINED_LAZ_PATH}")
print(f"Master-Tabelle: {MASTER_EXCEL_PATH}")
print(f"Exportierte Bäume: {len(export_ids)}")
print(f"Exportierte Punkte: {written_points:,}")

Höhenbereich: 8.01 bis 48.07 m

Höhenklassen:
Klasse  1: 8.01–10.68 m | 65 Bäume
Klasse  2: 10.68–13.35 m | 65 Bäume
Klasse  3: 13.35–16.02 m | 72 Bäume
Klasse  4: 16.02–18.69 m | 49 Bäume
Klasse  5: 18.69–21.36 m | 37 Bäume
Klasse  6: 21.36–24.03 m | 18 Bäume
Klasse  7: 24.03–26.70 m | 15 Bäume
Klasse  8: 26.70–29.38 m | 10 Bäume
Klasse  9: 29.38–32.05 m | 13 Bäume
Klasse 10: 32.05–34.72 m | 3 Bäume
Klasse 11: 34.72–37.39 m | 7 Bäume
Klasse 12: 37.39–40.06 m | 3 Bäume
Klasse 13: 40.06–42.73 m | 2 Bäume
Klasse 14: 42.73–45.40 m | 1 Bäume
Klasse 15: 45.40–48.07 m | 2 Bäume

Bäume für den Export: 362
Saubere Excel-Tabelle gespeichert unter:
Z:\Ghana\RCT_outputs\BO1_preseg_RCT\output\rct\segmented\BO1_Trees_Inventory\KO4_tree_inventory_master.xlsx
Block 183 verarbeitet | 175,419,535 Punkte geschrieben

Export abgeschlossen
LAZ-Datei: Z:\Ghana\RCT_outputs\BO1_preseg_RCT\output\rct\segmented\BO1_Trees_Inventory\KO4_Trees_Inventory.laz
Master-Tabelle: Z:\Ghana\RCT_outputs\BO1_preseg_RCT\outp

Der Code übernimmt die automatisch und manuell bestätigten Baumzuordnungen, verknüpft sie über Plot-ID und Stemmap-Tag mit den Inventardaten und exportiert nur Bäume mit eindeutig gefundenen Inventarinformationen. Anschließend werden die Baumhöhen in 15 gleich breite Höhenintervalle eingeteilt. Die jeweilige Height_Class wird sowohl in der Master-CSV als auch als zusätzliche Punktdimension in der gemeinsamen LAZ-Datei gespeichert, sodass die Bäume später nach PredInstance oder Height_Class gefiltert werden können.

In [36]:
from pathlib import Path
import pandas as pd

INVENTORY_PATH = Path(
    r"Z:\Ghana\field data\Field Data Request.xlsx"
)

inventory_raw = pd.read_excel(
    INVENTORY_PATH,
    sheet_name="Data"
)

# Spaltennamen etwas bereinigen
inventory_raw.columns = (
    inventory_raw.columns.astype(str)
    .str.replace("\n", " ", regex=False)
    .str.strip()
)

print(f"Inventarzeilen: {len(inventory_raw)}")
print(f"Inventarspalten: {len(inventory_raw.columns)}")

print("\nSpaltennamen:")
for column in inventory_raw.columns:
    print(column)

display(inventory_raw.head(10))

Inventarzeilen: 2648
Inventarspalten: 20

Spaltennamen:
Date
Ecozone
Site
PID
PT
Subplot
Tag
Tnb
Species
COR N
COR W
ELEV
WD
dbh (cm)
d2
H (m)
H Mode
Mass kg C
Status
Life Form


,Date,Ecozone,Site,PID,PT,Subplot,Tag,Tnb,Species,COR N,COR W,ELEV,WD,dbh (cm),d2,H (m),H Mode,Mass kg C,Status,Life Form
0,2023-02-21,WE,Ankasa Conservation Area,ANK-01,MP,1,1,NaN,Trichoscypha arborea (A.Chev.) A.Chev.,5.26824,2.69409,168,0.644,17.1,NaN,13.529889,NaN,63.231415,SDT,Tree
1,2023-02-21,WE,Ankasa Conservation Area,ANK-01,MP,1,2,NaN,Drypetes aylmeri Hutch. & Dalz.,5.26822,2.69412,168,0.667,13.0,NaN,11.014916,PH,33.439798,Alive,Tree
2,2023-02-21,WE,Ankasa Conservation Area,ANK-01,MP,1,3,NaN,Uapaca guineensis Muell.Arg.,5.26823,2.69414,168,0.612,62.0,NaN,33.797349,PH,1937.908183,Alive,Tree
3,2023-02-21,WE,Ankasa Conservation Area,ANK-01,MP,1,4,NaN,Anthonotha fragrans (Bak.f.) Excell & Hillc.,5.26822,2.69415,168,0.529,16.0,NaN,12.868343,PH,46.553765,Alive,Tree
4,2023-02-21,WE,Ankasa Conservation Area,ANK-01,MP,1,5,NaN,Cassipourea afzelii (Oliv.) Alston,5.26818,2.69415,168,0.645,12.8,NaN,10.888745,PH,31.047045,Alive,Tree
5,2023-02-21,WE,Ankasa Conservation Area,ANK-01,MP,1,6,NaN,Pentadesma butyracea Sabine,5.26817,2.69414,167,0.806,25.1,NaN,18.064771,PH,235.472585,Alive,Tree
6,2023-02-21,WE,Ankasa Conservation Area,ANK-01,MP,1,7,NaN,Berlinia tomentella Keay,5.26812,2.69409,167,0.645,27.9,NaN,19.542986,PH,251.467923,Alive,Tree
7,2023-02-21,WE,Ankasa Conservation Area,ANK-01,MP,1,8,NaN,Heritiera utilis (Sprague) Sprague,5.26812,2.69409,167,0.575,39.1,NaN,24.942292,PH,551.188955,Alive,Tree
8,2023-02-21,WE,Ankasa Conservation Area,ANK-01,MP,1,9,NaN,Vitex micrantha Gurke,5.26819,2.69407,167,0.454,22.3,NaN,16.531328,PH,97.897530,Alive,Tree
9,2023-02-21,WE,Ankasa Conservation Area,ANK-01,MP,1,10,NaN,Drypetes leonensis Pax,5.2682,2.69404,167,0.650,43.0,NaN,26.644013,PH,797.709719,Alive,Tree
